In [ ]:
import pandas as pd
import geopandas as gpd
import topojson as tp

In [ ]:
import pandas as pd
import geopandas as gpd
import topojson as tp

# 1. Load your CSV which has ZIPCODE, Year, and drug columns
latest_df = pd.read_csv(
    "../../reports/deidentified_overdose_201201202408_zips_0311.csv"
)
drug_cols = [
    "Methamphetamine",
    "Heroin",
    "Cocaine",
    "Fentanyl",
    "Alcohol",
    "Prescription.opioids",
    "Any Opioids",
    "Benzodiazepines",
    "Others",
    "Any Drugs",
]

# 2. Prepare data for aggregation
df_subset = latest_df[["ZIPCODE", "Year"] + drug_cols].copy()
# Remove rows with missing Year to avoid NaNs in aggregation
df_subset = df_subset[df_subset["Year"].notnull()]

# 3. Aggregate by ZIPCODE & Year
df_agg = df_subset.groupby(["ZIPCODE", "Year"])[drug_cols].sum().reset_index()

# 4. Create "All" rows by aggregating across all years for each ZIPCODE
df_all_years = df_subset.groupby("ZIPCODE")[drug_cols].sum().reset_index()
df_all_years["Year"] = "All"

# 5. Combine per-year data with the all-year rows
df_combined = pd.concat([df_agg, df_all_years], ignore_index=True)
# Rename columns (e.g. "Alcohol" -> "Alcohol_Count")
df_combined.columns = ["ZIPCODE", "Year"] + [f"{col}_Count" for col in drug_cols]

# 6. Load ZIP code geometry and simplify it
zip_gdf = gpd.read_file("../../data/zipcodes.geojson")
topo = tp.Topology(zip_gdf, prequantize=False)
simple = topo.toposimplify(0.001).to_gdf()
# Ensure ZIPCODE is a string and strip extra spaces if any
simple["ZIPCODE"] = simple["ZIPCODE"].astype(str).str.strip()
df_combined["ZIPCODE"] = df_combined["ZIPCODE"].astype(str).str.strip()

# 7. Build a complete combination table:
#    All zipcodes (from geometry), all years (from df_combined), and all overdose types.
all_zipcodes = simple["ZIPCODE"].unique()
all_years = df_combined["Year"].unique()  # includes actual years and "All"
overdose_types = [f"{col}_Count" for col in drug_cols]

# Create complete combinations using a MultiIndex
complete_combos = pd.MultiIndex.from_product(
    [all_zipcodes, all_years, overdose_types],
    names=["ZIPCODE", "Year", "Overdose_Type"],
).to_frame(index=False)

# 8. Convert your aggregated data (df_combined) to long format
df_long_existing = df_combined.melt(
    id_vars=["ZIPCODE", "Year"],
    value_vars=overdose_types,
    var_name="Overdose_Type",
    value_name="Overdose_Count",
)

# 9. Merge the complete combinations with your aggregated long data.
df_long_complete = complete_combos.merge(
    df_long_existing, on=["ZIPCODE", "Year", "Overdose_Type"], how="left"
)
# Fill missing counts with 0 (i.e. if a combination has no data, set to 0)
df_long_complete["Overdose_Count"] = df_long_complete["Overdose_Count"].fillna(0)

# 10. Build a complete combination table: all ZIPCODEs, all Years, all Overdose_Types.
all_zipcodes = simple["ZIPCODE"].unique()
all_years = df_combined["Year"].unique()  # includes actual years and "All"
overdose_types = [f"{col}_Count" for col in drug_cols]

# Create complete combinations using a MultiIndex
complete_combos = pd.MultiIndex.from_product(
    [all_zipcodes, all_years, overdose_types],
    names=["ZIPCODE", "Year", "Overdose_Type"],
).to_frame(index=False)

# 11. Convert your aggregated data (df_combined) to long format.
df_long_existing = df_combined.melt(
    id_vars=["ZIPCODE", "Year"],
    value_vars=overdose_types,
    var_name="Overdose_Type",
    value_name="Overdose_Count",
)

# 12. Merge the complete combinations with your aggregated long data.
df_long_complete = complete_combos.merge(
    df_long_existing, on=["ZIPCODE", "Year", "Overdose_Type"], how="left"
)
df_long_complete["Overdose_Count"] = df_long_complete["Overdose_Count"].fillna(0)

# 13. Instead of merging geometry, assign it via a mapping.
# Drop duplicates to ensure unique ZIPCODE keys.
geometry_map = simple.drop_duplicates(subset=["ZIPCODE"]).set_index("ZIPCODE")[
    "geometry"
]
df_long_complete["geometry"] = df_long_complete["ZIPCODE"].map(geometry_map)

# 14. Add composite key: e.g. "90001_2012_Alcohol_Count"
df_long_complete["composite_key"] = (
    df_long_complete["ZIPCODE"].astype(str)
    + "_"
    + df_long_complete["Year"].astype(str)
    + "_"
    + df_long_complete["Overdose_Type"]
)

df_long_complete["zipodtype"] = (
    df_long_complete["ZIPCODE"].astype(str) + "_" + df_long_complete["Overdose_Type"]
)

df_long_complete["ZIPCODE"] = df_long_complete["ZIPCODE"] + "_zip"

# 15. Convert to a GeoDataFrame and save as GeoJSON.
df_long_complete_gdf = gpd.GeoDataFrame(
    df_long_complete, geometry="geometry", crs=simple.crs
)

In [ ]:
df_long_complete_gdf.to_file(
    "zip_overdose_with_all_years_complete_zipstr_zipodtype.geojson",
    driver="GeoJSON",
)

In [ ]:
latest_df = pd.read_csv(
    "../../reports/deidentified_overdose_201201202408_zips_0311.csv"
)